In [ ]:
from IPython.display import display, HTML
import re, subprocess, threading, time, socket

# 1. Install ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt

# 2. Download SDXL 1.0 Base Model
print("Downloading SDXL 1.0 Base Model...")
!wget -q -O models/checkpoints/sd_xl_base_1.0.safetensors https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors

# 3. Download Real-ESRGAN x4 Plus (Upscaler)
print("Downloading Real-ESRGAN x4 Plus Model...")
!wget -q -O models/upscale_models/RealESRGAN_x4plus.pth https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth

# 4. Download VAE
print("Downloading SDXL VAE...")
!wget -q -O models/vae/sdxl_vae.safetensors https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors

# 5. Cloudflare tunnel function
def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if sock.connect_ex(('127.0.0.1', port)) == 0:
            sock.close()
            break
        sock.close()
    print("\nComfyUI is running. Starting Cloudflare tunnel...")
    # stderr=STDOUT so we catch output from both streams
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    for line in p.stdout:
        l = line.decode(errors='ignore')
        m = re.search(r'https?://[a-zA-Z0-9-]+\.trycloudflare\.com', l)
        if m:
            url = m.group(0)
            print("\n=======================================================")
            print(f"\u2705 YOUR SERVER URL: {url}")
            display(HTML(f'<a href="{url}" target="_blank">{url}</a>'))
            print("=======================================================\n")
            break

# 6. Install Cloudflared
!wget -q -c -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# 7. Start ComfyUI and Tunnel
threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server --enable-cors-header "*"